In [29]:
import os

from dotenv import load_dotenv
from training.common import FeedID, Operator
from training.data_processing import feed_message_to_vehicle_position_dataframe, join_static_data_on_rt_vehicle_positions
from training.gtfs import load_gtfs_rt_immediately
from training.static_data import StaticData
load_dotenv()


folder_path = "../../../../data"
model_path = "../../../../models"


GTFS_RT_API_KEY = os.environ.get("GTFS_REGIONAL_RT_API_KEY", "")
if GTFS_RT_API_KEY == "":
    raise Exception("GTFS_REGIONAL_RT_API_KEY environment variable not set")

gtfs_static_data = StaticData.load_static_data(f"{folder_path}/gtfs-static/data-tmp")



In [41]:
gtfs_vehicle_positions = load_gtfs_rt_immediately(
    operator=Operator.SL, 
    feedId=FeedID.VehiclePositions, 
    api_key= GTFS_RT_API_KEY,
)

buses_df = feed_message_to_vehicle_position_dataframe(gtfs_vehicle_positions)
buses_df = join_static_data_on_rt_vehicle_positions(gtfs_static_data, buses_df)

print(buses_df.dtypes)

buses_df.head()

id                                       Int64
trip_id                         string[python]
timestamp                                int64
vehicle_latitude                       float64
vehicle_longitude                      float64
vehicle_bearing                        float64
vehicle_odometer                       float64
vehicle_speed                          float64
vehicle_congestion_level                 int64
vehicle_occupancy_percentage             int64
vehicle_occupancy_status                 int64
stop_id                                 object
current_status                           int64
current_stop_sequence                    int64
route_id                                object
service_id                               Int64
trip_headsign                   string[python]
direction_id                           float64
shape_id                                 Int64
agency_id                       string[python]
route_short_name                string[python]
route_long_na

,id,trip_id,timestamp,vehicle_latitude,vehicle_longitude,vehicle_bearing,vehicle_odometer,vehicle_speed,vehicle_congestion_level,vehicle_occupancy_percentage,...,route_id,service_id,trip_headsign,direction_id,shape_id,agency_id,route_short_name,route_long_name,route_type,route_desc
0,55801768216479866,14010000711690043,1768216480,59.744625,18.366974,48.0,0.0,9.2,0,0,...,9011001067700000,268,<NA>,0.0,1014010000711688276,14010000000001001,677,<NA>,700,blåbuss
1,23311768216400478,14010000684355655,1768216480,59.236656,18.101954,0.0,0.0,-0.3,0,0,...,9011001004300000-43X,81,<NA>,1.0,4014010000684354191,14010000000001001,43X,<NA>,100,Pendeltåg
2,1221768216477282,14010100702399172,1768216480,59.330673,18.059980,205.0,0.0,-0.3,0,0,...,9011001001900000,57,<NA>,0.0,3014010000537447983,14010000000001001,19,Gröna linjen,401,tunnelbanans gröna linje
3,56081768216479843,14010000708110289,1768216480,59.759327,18.699602,70.0,0.0,0.0,0,0,...,9011001065600000,38,<NA>,0.0,1014010000313737853,14010000000001001,656,<NA>,700,<NA>
4,15401768216479842,14010000702275715,1768216480,59.328754,18.060959,155.0,0.0,0.0,0,0,...,9011001005300000,209,<NA>,1.0,1014010000660243247,14010000000001001,53,<NA>,700,<NA>


In [51]:
buses_df["route_short_name"].value_counts()

route_short_name
30     18
4      15
14     15
41     13
176    13
       ..
689     1
532     1
542     1
124     1
548     1
Name: count, Length: 335, dtype: Int64

In [42]:
STAM_BUSES_ROUTE_SET = {"9011001000100000", 
                    "9011001000200000",
                    "9011001000300000",
                    "9011001000400000"}

In [43]:
import datetime

from training.gtfs import list_gtfs_files, load_gtfs_frame_from_files

now = datetime.datetime.now(datetime.timezone.utc)
ten_minute_files = list_gtfs_files(
    base_path=f"{folder_path}/gtfs-rt/data-tmp/sl/TripUpdates",
    start_dt=now - datetime.timedelta(minutes=10),
    end_dt=now
)

gtfs_feed_df = load_gtfs_frame_from_files(ten_minute_files, gtfs_static_data)

print(gtfs_feed_df.dtypes)

gtfs_feed_df.head()

id                                Int64
trip_id                  string[python]
start_date               datetime64[ns]
schedule_relationship             int64
vehicle_id                        Int64
stop_time_updates                object
timestamp                         int64
route_id                         object
service_id                        Int64
trip_headsign            string[python]
direction_id                    float64
shape_id                          Int64
agency_id                string[python]
route_short_name         string[python]
route_long_name          string[python]
route_type                        Int64
route_desc               string[python]
dtype: object


,id,trip_id,start_date,schedule_relationship,vehicle_id,stop_time_updates,timestamp,route_id,service_id,trip_headsign,direction_id,shape_id,agency_id,route_short_name,route_long_name,route_type,route_desc
0,14010517667005976,14010000712052634,2026-01-12,0,9031001001004988,"[{'stop_sequence': 27, 'stop_id': '90220010504...",1768215882,9011001095200000,3,<NA>,0.0,1014010000712051577,14010000000001001,952,<NA>,700,Närtrafiken
1,14010517667827341,14010000712233272,2026-01-12,0,9031008000500547,"[{'stop_sequence': 16, 'stop_id': '90220010001...",1768215882,9011008001400000,371,<NA>,1.0,6014010000528814783,14010000000002071,14,<NA>,1000,Waxholmsbolaget
2,14010517621332965,14010000708107752,2026-01-12,0,9031001004505140,"[{'stop_sequence': 46, 'stop_id': '90220010645...",1768215882,9011001064500000,38,<NA>,1.0,1014010000241549329,14010000000001001,645,<NA>,700,<NA>
3,14010517213592090,14010000684355591,2026-01-12,0,9031001004302327,"[{'stop_sequence': 20, 'stop_id': '90220010062...",1768215882,9011001004300000-43X,81,<NA>,1.0,4014010000684354191,14010000000001001,43X,<NA>,100,Pendeltåg
4,14010517563279893,14010000708170497,2026-01-12,0,9031001003003068,"[{'stop_sequence': 62, 'stop_id': '90220010757...",1768215882,9011001078300000,38,<NA>,0.0,1014010000708164357,14010000000001001,783,<NA>,700,<NA>


In [44]:

from training.data_pipeline import lag_times
from training.data_processing import explode_to_stops_with_join_static, join_static_data_on_rt_trip_updates

trip_updates_df = gtfs_feed_df[gtfs_feed_df["route_id"].isin(STAM_BUSES_ROUTE_SET)]
trip_updates_df = join_static_data_on_rt_trip_updates(gtfs_static_data, trip_updates_df)
trip_updates_df = explode_to_stops_with_join_static(gtfs_static_data, trip_updates_df)
trip_updates_df = lag_times(trip_updates_df)


# Only keep the last line stop_sequence per trip_id
# multiindex: trip_id	stop_sequence	
trip_updates_df = trip_updates_df.sort_values(['trip_id', 'stop_sequence'])
trip_updates_df = trip_updates_df.groupby(level='trip_id').last()

trip_updates_df = trip_updates_df.reset_index()

print(trip_updates_df["trip_id"])
print(trip_updates_df.dtypes)

trip_updates_df.head()

arrival_time            datetime64[ns]
arrival_time_planned    datetime64[ns]
dtype: object
0     14010000655862772
1     14010000664227999
2     14010000664228032
3     14010000664228195
4     14010000664228228
5     14010000664228696
6     14010000664229038
7     14010000669012473
8     14010000672069744
9     14010000694062320
10    14010000702275649
11    14010000702275795
12    14010000702276053
13    14010000702276204
14    14010000702342480
15    14010000702343008
16    14010000707651582
17    14010000707651616
18    14010000707651690
19    14010000707651718
20    14010000707651760
21    14010000707651793
22    14010000707651903
23    14010000707651964
24    14010000707652009
25    14010000707652044
26    14010000707652077
27    14010000707652120
28    14010000707652149
29    14010000707653343
30    14010000710273405
31    14010000710273444
32    14010000710991107
33    14010000710993890
34    14010000710993933
35    14010100669011999
36    14010100669012126
37    14010100669012

,trip_id,id,start_date,schedule_relationship,vehicle_id,timestamp,route_id,service_id,trip_headsign,direction_id,...,drop_off_booking_rule_id,arrival_time_seconds_since_midnight,departure_time_seconds_since_midnight,arrival_time_planned,departure_time_planned,arrival_time_late,departure_time_late,arrival_time_prev,arrival_time_planned_prev,arrival_time_late_prev
0,14010000655862772,14010517577179983,2026-01-12,0,9031001001001558,1768216452,9011001000300000,12,<NA>,1.0,...,NaN,0 days 12:31:00,0 days 12:31:00,2026-01-12 12:31:00,2026-01-12 12:31:00,-1 days +23:57:39,-1 days +23:57:39,2026-01-12 12:27:24,2026-01-12 12:24:12,0 days 00:03:12
1,14010000664227999,14010517577201964,2026-01-12,0,9031001001007157,1768216226,9011001000300000,12,<NA>,1.0,...,NaN,0 days 12:10:00,0 days 12:10:00,2026-01-12 12:10:00,2026-01-12 12:10:00,-1 days +23:59:13,-1 days +23:59:13,2026-01-12 12:08:00,2026-01-12 12:03:12,0 days 00:04:48
2,14010000664228032,14010517576959244,2026-01-12,0,9031001001001555,1768216452,9011001000300000,12,<NA>,0.0,...,NaN,0 days 12:09:00,0 days 12:09:00,2026-01-12 12:09:00,2026-01-12 12:09:00,0 days 00:09:53,0 days 00:10:34,2026-01-12 12:16:32,2026-01-12 12:01:33,0 days 00:14:59
3,14010000664228195,14010517577217825,2026-01-12,0,9031001001001550,1768216452,9011001000300000,12,<NA>,1.0,...,NaN,0 days 12:20:00,0 days 12:20:00,2026-01-12 12:20:00,2026-01-12 12:20:00,-1 days +23:57:33,-1 days +23:57:33,2026-01-12 12:16:17,2026-01-12 12:13:12,0 days 00:03:05
4,14010000664228228,14010517576954344,2026-01-12,0,9031001001001549,1768216452,9011001000300000,12,<NA>,0.0,...,NaN,0 days 12:19:00,0 days 12:19:00,2026-01-12 12:19:00,2026-01-12 12:19:00,0 days 00:07:03,0 days 00:07:03,2026-01-12 12:23:59,2026-01-12 12:11:33,0 days 00:12:26


In [45]:
from training.model_training import load_route_model

from xgboost import XGBRegressor

xgbModels:dict[str, XGBRegressor] = {}

for route in STAM_BUSES_ROUTE_SET:
    xgbModels[route] = load_route_model(route, model_path=model_path)

In [46]:

import pandas as pd

from training.data_pipeline import create_X_from_df


def do_inference_on_route(route_id: str, model: XGBRegressor, data: pd.DataFrame) -> pd.DataFrame:
    filtered_data = data[data["route_id"] == route_id].copy()
    filtered_data["arrival_time_late_prev"] = filtered_data["arrival_time_late_prev"].fillna(
        pd.to_timedelta(1, unit="s")
    )

    x_data = create_X_from_df(filtered_data)

    late_preds = model.predict(x_data)

    # ensure datetime type + add timedelta seconds
    filtered_data["arrival_time_estimate"] = (
        pd.to_datetime(filtered_data["arrival_time_planned"])
        + pd.to_timedelta(pd.Series(late_preds), unit="s")
    )

    return filtered_data

inferences = []

for route_id in STAM_BUSES_ROUTE_SET:
    inferences.append(do_inference_on_route(route_id=route_id, model=xgbModels[route_id], data=trip_updates_df))

inference_df = pd.concat(inferences)

print(inference_df.dtypes)

inference_df.head()

trip_id                                   string[python]
id                                                 Int64
start_date                                datetime64[ns]
schedule_relationship                              int64
vehicle_id                                         Int64
timestamp                                          int64
route_id                                          object
service_id                                         Int64
trip_headsign                             string[python]
direction_id                                     float64
shape_id                                           Int64
agency_id                                 string[python]
route_short_name                          string[python]
route_long_name                           string[python]
route_type                                         Int64
route_desc                                string[python]
route_id_trip                             string[python]
service_id_trip                

,trip_id,id,start_date,schedule_relationship,vehicle_id,timestamp,route_id,service_id,trip_headsign,direction_id,...,arrival_time_seconds_since_midnight,departure_time_seconds_since_midnight,arrival_time_planned,departure_time_planned,arrival_time_late,departure_time_late,arrival_time_prev,arrival_time_planned_prev,arrival_time_late_prev,arrival_time_estimate
0,14010000655862772,14010517577179983,2026-01-12,0,9031001001001558,1768216452,9011001000300000,12,<NA>,1.0,...,0 days 12:31:00,0 days 12:31:00,2026-01-12 12:31:00,2026-01-12 12:31:00,-1 days +23:57:39,-1 days +23:57:39,2026-01-12 12:27:24,2026-01-12 12:24:12,0 days 00:03:12,2026-01-12 12:30:58.273663521
1,14010000664227999,14010517577201964,2026-01-12,0,9031001001007157,1768216226,9011001000300000,12,<NA>,1.0,...,0 days 12:10:00,0 days 12:10:00,2026-01-12 12:10:00,2026-01-12 12:10:00,-1 days +23:59:13,-1 days +23:59:13,2026-01-12 12:08:00,2026-01-12 12:03:12,0 days 00:04:48,2026-01-12 12:14:42.231140137
2,14010000664228032,14010517576959244,2026-01-12,0,9031001001001555,1768216452,9011001000300000,12,<NA>,0.0,...,0 days 12:09:00,0 days 12:09:00,2026-01-12 12:09:00,2026-01-12 12:09:00,0 days 00:09:53,0 days 00:10:34,2026-01-12 12:16:32,2026-01-12 12:01:33,0 days 00:14:59,2026-01-12 12:21:45.184631348
3,14010000664228195,14010517577217825,2026-01-12,0,9031001001001550,1768216452,9011001000300000,12,<NA>,1.0,...,0 days 12:20:00,0 days 12:20:00,2026-01-12 12:20:00,2026-01-12 12:20:00,-1 days +23:57:33,-1 days +23:57:33,2026-01-12 12:16:17,2026-01-12 12:13:12,0 days 00:03:05,2026-01-12 12:21:24.569267273
4,14010000664228228,14010517576954344,2026-01-12,0,9031001001001549,1768216452,9011001000300000,12,<NA>,0.0,...,0 days 12:19:00,0 days 12:19:00,2026-01-12 12:19:00,2026-01-12 12:19:00,0 days 00:07:03,0 days 00:07:03,2026-01-12 12:23:59,2026-01-12 12:11:33,0 days 00:12:26,2026-01-12 12:29:50.388549805


In [48]:
buses_df = buses_df.set_index("trip_id")
inference_df = inference_df.reset_index()
inference_df = inference_df.set_index("trip_id")

inference_df = inference_df.join(buses_df, on="trip_id", rsuffix="_trip", how="inner")

inference_df.head()

,index,id,start_date,schedule_relationship,vehicle_id,timestamp,route_id,service_id,trip_headsign,direction_id,...,route_id_trip,service_id_trip,trip_headsign_trip,direction_id_trip,shape_id_trip,agency_id_trip,route_short_name_trip,route_long_name_trip,route_type_trip,route_desc_trip
trip_id,,,,,,,,,,,,,,,,,,,,,
14010000655862772,0,14010517577179983,2026-01-12,0,9031001001001558,1768216452,9011001000300000,12,<NA>,1.0,...,9011001000300000,12,<NA>,1.0,1014010000571984641,14010000000001001,3,<NA>,700,blåbuss
14010000664228032,2,14010517576959244,2026-01-12,0,9031001001001555,1768216452,9011001000300000,12,<NA>,0.0,...,9011001000300000,12,<NA>,0.0,1014010000573107346,14010000000001001,3,<NA>,700,blåbuss
14010000664228195,3,14010517577217825,2026-01-12,0,9031001001001550,1768216452,9011001000300000,12,<NA>,1.0,...,9011001000300000,12,<NA>,1.0,1014010000571984641,14010000000001001,3,<NA>,700,blåbuss
14010000664228228,4,14010517576954344,2026-01-12,0,9031001001001549,1768216452,9011001000300000,12,<NA>,0.0,...,9011001000300000,12,<NA>,0.0,1014010000573107346,14010000000001001,3,<NA>,700,blåbuss
14010000664228696,5,14010517577200128,2026-01-12,0,9031001001007217,1768216452,9011001000300000,12,<NA>,1.0,...,9011001000300000,12,<NA>,1.0,1014010000571984641,14010000000001001,3,<NA>,700,blåbuss


In [52]:
inference_df[["vehicle_id", "route_short_name", "stop_id", "arrival_time_planned", "arrival_time_estimate",
            
            "arrival_time_planned_prev",
            "arrival_time_late_prev",]]

,vehicle_id,route_short_name,stop_id,arrival_time_planned,arrival_time_estimate,arrival_time_planned_prev,arrival_time_late_prev
trip_id,,,,,,,
14010000655862772,9031001001001558,3,9022001010406001,2026-01-12 12:31:00,2026-01-12 12:30:58.273663521,2026-01-12 12:24:12,0 days 00:03:12
14010000664228032,9031001001001555,3,9022001050800003,2026-01-12 12:09:00,2026-01-12 12:21:45.184631348,2026-01-12 12:01:33,0 days 00:14:59
14010000664228195,9031001001001550,3,9022001010406001,2026-01-12 12:20:00,2026-01-12 12:21:24.569267273,2026-01-12 12:13:12,0 days 00:03:05
14010000664228228,9031001001001549,3,9022001050800003,2026-01-12 12:19:00,2026-01-12 12:29:50.388549805,2026-01-12 12:11:33,0 days 00:12:26
14010000664228696,9031001001007217,3,9022001010406001,2026-01-12 12:41:00,2026-01-12 12:40:35.177486420,2026-01-12 12:34:12,0 days 00:02:49
14010000664229038,9031001001007162,3,9022001010406001,2026-01-12 12:51:00,2026-01-12 12:52:15.136085510,2026-01-12 12:44:12,0 days 00:03:03
14010000702275649,9031001001007161,3,9022001050800003,2026-01-12 12:30:00,2026-01-12 12:29:42.740806580,2026-01-12 12:21:55,0 days 00:02:48
14010000702275795,9031001001007203,3,9022001050800003,2026-01-12 12:40:00,NaT,2026-01-12 12:31:55,0 days 00:06:46
14010000702276053,9031001001007177,3,9022001050800003,2026-01-12 12:50:00,NaT,2026-01-12 12:41:55,0 days 00:04:35
